*Notebook 07 in `notebooks/arcs/`. Parameters (pre-registered): `analysis/arcs/config.yaml`; decisions and run notes: `analysis/arcs/LOG.md`. The static page `site/arcs/index.html` is built from the same saved outputs.*

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # notebooks/arcs -> repo root
sys.path.insert(0, str(REPO))

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

from analysis.arcs import s07_primary as step

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Step 7 · PRIMARY ANALYSIS. Compared to what (2): is the arc associated with profit?

**The one pre-registered test.** In the primary sample (not likely franchise, arc-ready, N = 834), we relate
each of the two primary descriptors (D1 volatility, D2 ending minus start) to the outcome,
log(worldwide gross / budget).

**Controls on both sides (Frisch-Waugh).** The outcome and the descriptor are each regressed on:
- genre × decade cells (cells with fewer than 5 films pooled);
- log budget;
- mean sentences per scene.

We then take the Spearman correlation of the two residuals. That is the association left once the controls
are held fixed.

**Compared to what.**
- We permute the residualised outcome 1,000 times **within** genre × decade cells, which keeps the cell
  structure, and recompute ρ each time.
- The two-sided empirical p is compared with the Bonferroni α = 0.025, because two descriptors are tested.
- A bootstrap of 2,000 film resamples gives the 95% CI; the residualisation is refitted in every resample.
- We also report the minimum detectable correlation. A null result then reads "we can rule out associations
  larger than about 0.1", not "there is none".

Reviews are not used here: they come after release. Everything above was fixed in `config.yaml` before any
association was computed, and this step runs once.

In [ ]:
res = step.run()   # recomputes this step and saves its outputs

**Table.** Sample, controls, numbers of permutations and resamples, and power.

In [ ]:
display(step.tables(res)["setup"])

,item,value
0,instrument (fallback rule),sst2
1,films,834
2,outcome,log(worldwide gross / budget)
3,controls,"genre × decade cells + log_budget, mean_sentences_per_scene"
4,cells (pooled below 5 films),36
5,permutations within cells / bootstrap resamples,1000 / 2000
6,"minimum detectable |ρ| at 80% power, α = 0.05",0.097
7,… at α = 0.025 (Bonferroni),0.107
8,config hash,605801c43e


**Table.** The primary result: one row per descriptor.

In [ ]:
display(step.tables(res)["results"])

,descriptor,N,Spearman ρ (residualised),bootstrap 95% CI,null mean ± sd,z,empirical p,Bonferroni α,decision
0,D1 volatility,834,0.035,"[-0.033, +0.106]",+0.001 ± 0.035,0.97,0.3187,0.025,no evidence of an association
1,D2 ending minus start,834,0.078,"[+0.010, +0.153]",+0.000 ± 0.035,2.23,0.0280,0.025,no evidence of an association


**Figure.** D1: the 1,000 within-cell permutation correlations, the real one (blue) and ±r_min (dotted).

In [ ]:
step.figures(res)["null_D1"].show()

**Figure.** D2: the same for ending minus start.

In [ ]:
step.figures(res)["null_D2"].show()

**Figure.** D1: rank of the residualised descriptor against rank of the residualised outcome, one point per film (no names).

In [ ]:
step.figures(res)["ranks_D1"].show()

**Figure.** D2: the same.

In [ ]:
step.figures(res)["ranks_D2"].show()

**Plain-language reading** (generated from the saved numbers):

In [ ]:
display(Markdown(step.reading(res)))

Primary analysis on 834 films, instrument sst2. D1 volatility: ρ = +0.035 (95% CI -0.033 to +0.106), p = 0.319, z = +0.97: no evidence of an association at the pre-registered α = 0.025. The interval runs from -0.03 to +0.11, so associations larger than that in either direction are unlikely; with N = 834, |ρ| ≥ 0.107 would be detected with 80% power.  D2 ending minus start: ρ = +0.078 (95% CI +0.010 to +0.153), p = 0.028, z = +2.23: no evidence of an association at the pre-registered α = 0.025. It is a near miss: the bootstrap interval excludes zero and p is below 0.05 but above 0.025. Under the rule fixed before the analysis this is not a finding; at most it suggests an association no larger than about 0.15, to be checked for stability in Steps 9 and 10.  Two caveats carry over: the sentiment scores are NOT YET VALIDATED against human readers (Step 4), and the split-half reliability of D1 and D2 (Step 5) attenuates any true association towards zero.